# EDA – Future Realized Volatility Forecasting

This notebook supports the **main ML volatility-forecasting project**.
It focuses on:
- price series overview
- log returns
- realized-volatility features
- target distribution
- chronological train/validation/test split
- actual vs predicted
- residual analysis


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

project_root = Path('..')
cleaned = pd.read_csv(project_root / 'data/processed/cleaned_prices.csv', parse_dates=['date'])
features = pd.read_csv(project_root / 'data/processed/features_prices.csv', parse_dates=['date'])
ml_df = pd.read_csv(project_root / 'data/processed/ml_dataset.csv', parse_dates=['date'])
train_df = pd.read_csv(project_root / 'data/splits/train.csv', parse_dates=['date'])
val_df = pd.read_csv(project_root / 'data/splits/val.csv', parse_dates=['date'])
test_df = pd.read_csv(project_root / 'data/splits/test.csv', parse_dates=['date'])
test_metrics = pd.read_csv(project_root / 'data/results/test_metrics.csv')
test_predictions = pd.read_csv(project_root / 'data/results/test_predictions.csv', parse_dates=['date'])
cleaned.head()

## 1. Price series overview

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cleaned['date'], cleaned['close'])
plt.title('AAPL close price')
plt.xlabel('Date')
plt.ylabel('Close')
plt.tight_layout()
plt.show()

## 2. Log returns and realized-volatility features

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(features['date'], features['log_return'])
axes[0].set_title('Log returns')
axes[1].plot(features['date'], features['rv_20d_ann'], label='rv_20d_ann')
axes[1].plot(features['date'], features['ewma_vol_ann'], label='ewma_vol_ann', alpha=0.8)
axes[1].legend()
axes[1].set_title('Historical volatility features')
plt.tight_layout()
plt.show()

## 3. Target distribution

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(ml_df['future_20d_rv_ann'], bins=30)
plt.title('Target distribution: future_20d_rv_ann')
plt.xlabel('Future 20-day annualized realized volatility')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Chronological split illustration

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_df['date'], train_df['future_20d_rv_ann'], label='Train')
plt.plot(val_df['date'], val_df['future_20d_rv_ann'], label='Validation')
plt.plot(test_df['date'], test_df['future_20d_rv_ann'], label='Test')
plt.title('Chronological train/validation/test split')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Test-set model comparison

In [ ]:
test_metrics.sort_values('rmse')

## 6. Actual vs predicted for the selected model

In [ ]:
selected_pred = test_predictions.loc[test_predictions['is_selected_model'] == 1].copy()
selected_model_name = selected_pred['model_name'].iloc[0]

plt.figure(figsize=(7, 4))
plt.scatter(selected_pred['future_20d_rv_ann'], selected_pred['prediction'], alpha=0.6)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title(f'Actual vs predicted ({selected_model_name})')
plt.tight_layout()
plt.show()

## 7. Residual analysis

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(selected_pred['residual'], bins=30)
plt.title(f'Residual distribution ({selected_model_name})')
plt.xlabel('Residual')
plt.ylabel('Count')
plt.tight_layout()
plt.show()